# 8章 デザインとリファクタリング

## 8.1 プロジェクトのデザインと構造

## 8.2 機械プロジェクトの例

1. データのロード
2. データのクリーンアップと前処理を行い、MLに適した特徴量に変換
3. モデルのトレーニング
4. 検証データセットを使ったモデルの性能分析

## 8.2 コードのデザイン

### 8.2.1 モジュラーコード

**_モジュラーコード_**（モジュラー化されたコード）とは、**コードが独立した小さなパーツに分割されていること**を意味する。  
可能な限り、各関数やクラスには明確に定義された目的を1つ持たせること。その関数が行う1つの「こと」を考える。
  → 「この関数はデータの視覚化を担当する」というように、**短い文章で説明出来るものでなければならない**。

### 8.2.2 コードデザインの枠組み

以下は関数、クラス、メソッドを構築するための指針を挙げよう---Rachel TatmanによるKaggleに関する記事「Six steps to more professional data science code」（https://oreil.ly/Psa1F ）からの引用。

#### 関数名
関数の名前を決めることから始めよう。名前をつけることで、その関数に何をさせたいかという意図が明らかになる。良い名前の決め方については9章にて
#### 入力
関数のインターフェースは一貫していなければならないので、関数本体を書き始める前に、関数への入力を決めておくとよい。関数の引数にそれを指定する。インターフェースについては次節にて
#### ビヘイビア
関数やメソッドの本体には、その関数が実行する実際の処理が含まれる。これが関数の「**ビヘイビア**」であり、実際にやってもらいたいことである。前述したように、**各関数が行うことは1つだけ**にしておくのがよい
#### 出力
関数の出力は、_return_ 文を使って返すものであったり、ファイルに保存するデータであったりする。出力は関数のインターフェースの一部でもあるので、関数を書く前に出力についてよく考え、できるだけ変えないようにする

### 8.2.3 インターフェースと契約

モジュラーコードを構築する上で最重要といえるのが、システム内のコンポーネント間のインターフェースである。ある意味、個々の関数を記述する歳の出発点ともなる事柄である。  
入力として何を受け入れ、出力として何を返すのか、というところから始める。  
次の関数は、1章から何度か登場したものだが、CSVファイルと、落とす(dropする, 削除する)列のリストを入力として受け取り、pandasのDataFrameを出力として返す。

```python
def process_std_data(excel_file, columns_to_drop):
    df = pd.read_excel(excel_file)
    df = df.drop(columns_to_drop, axis=1)
    df = df.set_index("GeoAreaName").transpose()
    return df
```

一度入力と出力を決めたら、それを変更すべきではない。なぜなら、システムの他のコンポーネントがそれに依存しているかもしれないからだ。  
 → これは「**契約**（_contract_）」とも呼ばれる。**入力引数の数は、せいぜい3つか4つ程度と、少なめにしておくのがよい。これ以上の入力が必要な場合は、代わりに構成ファイル（コンフィギュレーションファイル）を使うことを検討する。**

### 8.2.4 結合度

コードを分割する場合、その分割されたコードが可能な限り互いに独立していることが重要。独立していないと、コードのある部分を変更すると、別の部分の変更が必要になり、プロジェクト全体の複雑さが増し、作業が難しくなる。  
**結合度**（カップリング）は、**関数やモジュール間の依存関係の強さを表す用語**。　　
2つの関数が  
密接合の場合：一方を変更すると、もう一方も（広範囲）に変更する必要がある  
疎密合の場合：一方を変更してもう一方は、あまり（あるいはまったく）変更する必要がない  
 → できるだけ結合度を低くするべきである。

コードデザインについて詳しく知りたい場合は「SOLID原則」について調べる。  
→ Read Pythonの記事（https://oreil.ly/oZN0y ）から始めるのが良い？

デザインパターンについては「The Refactoring Guru」（https://oreil.ly/HEV6u ）にPythonのデザインパターンに関する有用なガイドがある。

## 8.3 ノートブックからスケーラブルなスクリプトへ

### 8.3.1 なぜノートブックの代わりにスクリプトを使うのか

ノートブックの使い所
- 探索的にデータを見たい時
- 試験的にコードを試したい時
スクリプトの使い所
- 定常的なタスクとして処理したい時
- 上記としての運用をある程度以上の期間行いたい時

### 8.3.2 ノートブックからのスクリプト作成

例として、ノートブックから、繰り返し実行されている処理の行を抽出する

```python
df = pd.read_excel("SG_GEN_PARL.xlsx")
df = df.drop(["Goal", "Target", "Indicator", "SeriesCode", "SeriesDescription", "GeoAreaCode", "Reporting Type", "Sex", "Units",], axis=1)
df = df.set_index("GeoAreaName").transpose()
df.to_csv("women_in_parliament_processed.csv")
```

こうした行を関数化するために、全体的な目的や振る舞いを考える。

メモ  
- 目的：Excelファイルを取り込み、データを整形（カラムの削除、インデックス作成）、CSVファイルを出力する
- 関数の名前：`process_sdg_data`
- 関数の入力：Excelファイル
- 関数の出力：CSVファイル
- 入力ファイルや出力ファイルは変更の可能性があるので関数の引数とする

```python
def process_sdg_data(input_excel_file, drop_columns, output_csv_file):
    df = pd.read_excel(input_excel_file)
    df = pd.drop(drop_columns, axis=1)
    df = df.set_index("GeoAreaName").transpose()
    df.to_csv(output_csv_file)
```

次のステップとしてテストを追加する。  
データを表示したノートブックの行から、テスト項目をいくつか想定した。  

```python
import os
import pandas as pd

def test_process_sdg_data():
    test_filepath = "test_sdg_data.csv"

    process_sdg_data("SG_GEN_PARL.xlsx",
                    ["Goal", "Target", "Indicator", "SeriesCode",
                    "SeriesDescription", "GeoAreaCode", "Reporting Type",
                    "Sex", "Units"],
                    test_filepath)

    df = pd.read_csv(test_filepath)

    assert len(df) == 24
    assert len(df.columns) == 196

    # クリーンアップのステップ - このテストで作成されたファイルを削除
    os.remove(test_filepath)

```

ノートブックからスクリプトへの変換を支援するツール
- nbconvert；[リンク](https://nbconvert.readthedocs.io/en/latest/)
- Jupytext：[リンク](https://jupytext.org/)
- Kedro：[リンク](https://kedro.org/)

## 8.4 リファクタリング

### 8.4.1 リファクタリングの戦略

理想的な流れ：リファクタリングをする前に、該当のコードのテスト一式の準備をする。

### 8.4.2 リファクタリングの例

一般的なリファクタリングの原則
1. 変更を行う
2. テストを実行して何も壊れていないことを確認する
3. 変更を保存する（通常はVCS - Gitなど - にコミットする）

ここで重み付きの平均を求める関数を使って、上記のワークフローを進めてみる

```python
def weighted_mean(num_lst, weights):
    if not (num_lst or weights):
        return None
    running_total = 0
    for i in range(len(num_lst)):
        running_total += (num_lst[i] * weights[i])
    return (running_total/sum(weights))

```

そして、これが上の関数のテスト
```python
def test_weighted_mean():

    result = weighted_mean([1, 2, 4], [1, 2, 4])
    assert result == 3

    empty_list_result = weighted_mean([], [])
    assert not empty_list_result
```

リファクタリングの前にテストを実行し、全てが正しく機能していることを確認する。
```bash
$pytest test_weighted_mean.py 
=================== test session starts ===================
platform darwin -- Python 3.13.4, pytest-9.0.2, pluggy-1.6.0
rootdir: /Users/Shared/Learning/Books/books_Software_Engineering_for_Data_Scientists
configfile: pyproject.toml
plugins: anyio-4.12.0, Faker-38.2.0, typeguard-4.4.4
collected 1 item                                                                                                                                        

test_weighted_mean.py .                                                                                                                           [100%]

=================== 1 passed in 0.04s ===================
```


## 8.5 まとめ
プロジェクトの前・途中・完成後を通して、コードのデザインを意識することは重要だ。よくデザインされたコードは他人にも扱いやすく、久しぶりに戻っても作業を再開しやすい。

- 標準化 — 似たプロジェクトを繰り返すなら、再利用できるテンプレートを用意する。CookiecutterやKedroでセットアップを自動化できる。
- モジュール化 — 再利用・変更しやすいようコードを分割する。関数はまず目的とインタフェースを決めてから中身を書く。実験はJupyter、本番ではスクリプトへの移行を検討する。
- リファクタリング — 要件の変化や可読性・効率の改善で修正は避けられない。鍵はよいテストセット。小さく変更 → テストで確認 → 保存、を繰り返す。

コードのデザインは本番品質に欠かせないが、自分のコードが何をするかを関係者に伝えることも同じく重要だ。次章ではドキュメンテーションのベストプラクティスを扱う。